# Indonesian Stock ML Prediction System
## End-to-End Training and Evaluation Pipeline

This notebook demonstrates the complete pipeline for training and evaluating the stock prediction system.

## 1. Setup and Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from datetime import datetime
import os

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Import custom modules
from src.data.data_loader import IndonesianStockDataLoader, MacroDataLoader
from src.features.features import FeatureEngineer, get_feature_columns
from src.models.models import CombinedStockPredictor
from src.models.monte_carlo import MonteCarloSimulator, generate_price_forecast
from src.strategy.strategy import MLTradingStrategy
from src.backtest.backtest import WalkForwardBacktest, Backtester, PerformanceMetrics

print("✓ All modules imported successfully")

In [ ]:
# Load configuration
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"  Tickers: {len(config['data']['universe'])}")
print(f"  Start date: {config['data']['start_date']}")
print(f"  Horizons: {config['horizons']}")
print(f"  Target CAGR: {config['backtest']['target_cagr']*100}%")

## 2. Data Loading

In [ ]:
# Initialize data loaders
stock_loader = IndonesianStockDataLoader(
    tickers=config['data']['universe'],
    start_date=config['data']['start_date'],
    cache_dir=config['data']['cache_dir']
)

macro_loader = MacroDataLoader(
    index_ticker=config['data']['macro_tickers']['index'],
    forex_ticker=config['data']['macro_tickers']['forex'],
    start_date=config['data']['start_date'],
    cache_dir=config['data']['cache_dir']
)

print("Data loaders initialized")

In [ ]:
# Download stock data
print("Downloading stock data...")
stock_data_dict = stock_loader.download_all(use_cache=True)

print(f"\nDownloaded data for {len(stock_data_dict)} tickers")
for ticker, df in list(stock_data_dict.items())[:5]:
    print(f"  {ticker}: {len(df)} days, {df.index[0]} to {df.index[-1]}")

In [ ]:
# Download macro data
print("Downloading macro data...")
macro_data = macro_loader.get_all_macro_data(use_cache=True)

index_data = macro_data['index']
forex_data = macro_data['forex']

print(f"\nIndex data: {len(index_data)} days")
print(f"Forex data: {len(forex_data)} days")

## 3. Feature Engineering

In [ ]:
# Initialize feature engineer
feature_engineer = FeatureEngineer(
    technical_windows=config['features']['technical_windows'],
    momentum_windows=config['features']['momentum_windows'],
    volatility_windows=config['features']['volatility_windows'],
    volume_windows=config['features']['volume_windows'],
    shift_periods=config['features']['shift_periods']
)

print("Feature engineer initialized")

In [ ]:
# Generate features for a sample ticker
sample_ticker = list(stock_data_dict.keys())[0]
sample_data = stock_data_dict[sample_ticker]

print(f"Generating features for {sample_ticker}...")
sample_features = feature_engineer.create_all_features(
    sample_data,
    index_data,
    horizons=config['horizons']
)

print(f"\nGenerated {len(sample_features.columns)} columns")
print(f"Features shape: {sample_features.shape}")
print(f"\nSample columns:")
print(sample_features.columns.tolist()[:20])

In [ ]:
# Generate features for all tickers
print("Generating features for all tickers...")
all_features = []

for ticker, data in stock_data_dict.items():
    try:
        features = feature_engineer.create_all_features(
            data,
            index_data,
            horizons=config['horizons']
        )
        features['Ticker'] = ticker
        all_features.append(features)
        print(f"  ✓ {ticker}")
    except Exception as e:
        print(f"  ✗ {ticker}: {e}")

# Combine all features
features_df = pd.concat(all_features, axis=0)
features_df = features_df.sort_index()

print(f"\nCombined features shape: {features_df.shape}")
print(f"Date range: {features_df.index[0]} to {features_df.index[-1]}")

## 4. Model Training

In [ ]:
# Select prediction horizon (train separate models for each)
HORIZON = 21  # Start with 1-month horizon

print(f"Training models for {HORIZON}-day horizon")

# Prepare data
feature_cols = get_feature_columns(features_df)
target_direction_col = f'Direction_{HORIZON}d'
target_return_col = f'Future_Return_{HORIZON}d'

# Remove rows with missing targets
train_data = features_df[[*feature_cols, target_direction_col, target_return_col, 'Ticker']].dropna()

print(f"Training data shape: {train_data.shape}")
print(f"Features: {len(feature_cols)}")

In [ ]:
# Split data (time-series split)
train_size = int(len(train_data) * 0.8)

train_df = train_data.iloc[:train_size]
val_df = train_data.iloc[train_size:]

X_train = train_df[feature_cols]
y_train_dir = train_df[target_direction_col]
y_train_ret = train_df[target_return_col]

X_val = val_df[feature_cols]
y_val_dir = val_df[target_direction_col]
y_val_ret = val_df[target_return_col]

print(f"Train set: {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")
print(f"\nClass distribution (train):")
print(y_train_dir.value_counts(normalize=True))

In [ ]:
# Initialize and train models
predictor = CombinedStockPredictor(
    classifier_params={
        'model_type': config['models']['classification']['model_type'],
        'params': config['models']['classification']['params'],
        'calibration_method': config['models']['classification']['calibration_method']
    },
    regressor_params={
        'model_type': config['models']['regression']['model_type'],
        'params': config['models']['regression']['params'],
        'quantile_regression': config['models']['regression']['quantile_regression'],
        'quantiles': config['models']['regression']['quantiles']
    }
)

print("Training models...")
predictor.fit(
    X_train, y_train_dir, y_train_ret,
    X_val, y_val_dir, y_val_ret
)

## 5. Model Evaluation

In [ ]:
# Evaluate classification model
print("Classification Model Performance:")
print("="*50)

train_metrics = predictor.classifier.evaluate(X_train, y_train_dir)
val_metrics = predictor.classifier.evaluate(X_val, y_val_dir)

print("\nTrain Metrics:")
for metric, value in train_metrics.items():
    print(f"  {metric}: {value:.4f}")

print("\nValidation Metrics:")
for metric, value in val_metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Evaluate regression model
print("Regression Model Performance:")
print("="*50)

train_metrics = predictor.regressor.evaluate(X_train, y_train_ret)
val_metrics = predictor.regressor.evaluate(X_val, y_val_ret)

print("\nTrain Metrics:")
for metric, value in train_metrics.items():
    print(f"  {metric}: {value:.4f}")

print("\nValidation Metrics:")
for metric, value in val_metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Feature importance
importance_df = predictor.classifier.get_feature_importance()

print("\nTop 20 Most Important Features:")
print(importance_df.head(20))

# Plot
plt.figure(figsize=(12, 8))
plt.barh(importance_df['feature'].head(20), importance_df['importance'].head(20))
plt.xlabel('Importance')
plt.title('Top 20 Feature Importance (Classification Model)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Save Models

In [ ]:
# Save trained models
models_dir = config['output']['models_dir']
os.makedirs(models_dir, exist_ok=True)

classifier_path = os.path.join(models_dir, f'classifier_{HORIZON}d.pkl')
regressor_path = os.path.join(models_dir, f'regressor_{HORIZON}d.pkl')

predictor.save(classifier_path, regressor_path)

print(f"Models saved to:")
print(f"  Classifier: {classifier_path}")
print(f"  Regressor: {regressor_path}")

## 7. Backtesting

This section demonstrates a simplified backtest. For full walk-forward testing, see the separate backtesting script.

In [ ]:
# Generate predictions on validation set
predictions = predictor.predict(X_val)

# Create predictions DataFrame
predictions_df = pd.DataFrame({
    'date': val_df.index,
    'ticker': val_df['Ticker'],
    'probability': predictions['direction_proba'],
    'expected_return': predictions['expected_return']
})

print(f"Generated {len(predictions_df)} predictions")
print(predictions_df.head())

In [ ]:
# Initialize strategy
strategy = MLTradingStrategy(
    min_probability=config['strategy']['entry']['min_probability'],
    min_expected_return=config['strategy']['entry']['min_expected_return'],
    max_drawdown=config['strategy']['exit']['max_drawdown'],
    profit_target=config['strategy']['exit']['profit_target'],
    position_sizing_method=config['strategy']['position_sizing']['method'],
    target_volatility=config['strategy']['position_sizing']['target_volatility'],
    max_position_size=config['strategy']['position_sizing']['max_position_size'],
    max_positions=config['strategy']['max_positions'],
    commission=config['backtest']['costs']['commission'],
    slippage=config['backtest']['costs']['slippage']
)

print("Strategy initialized")
print(f"  Min probability: {strategy.min_probability}")
print(f"  Min expected return: {strategy.min_expected_return}")
print(f"  Max positions: {strategy.max_positions}")

## 8. Monte Carlo Simulation Example

In [ ]:
# Example Monte Carlo simulation for a ticker
example_ticker = list(stock_data_dict.keys())[0]
example_data = stock_data_dict[example_ticker]

current_price = example_data['Adj Close'].iloc[-1]
returns = example_data['Adj Close'].pct_change().dropna()
volatility = returns.tail(60).std() * np.sqrt(252)

# Use model prediction
latest_features = sample_features[feature_cols].dropna().iloc[[-1]]
pred = predictor.predict(latest_features)
expected_return = pred['expected_return'][0]

print(f"\nMonte Carlo Simulation for {example_ticker}")
print(f"Current Price: {current_price:.2f}")
print(f"Expected Return: {expected_return*100:.2f}%")
print(f"Volatility: {volatility*100:.2f}%")

# Run simulation
mc_results = generate_price_forecast(
    ticker=example_ticker,
    current_price=current_price,
    expected_return=expected_return,
    volatility=volatility,
    horizon_days=HORIZON,
    n_simulations=config['monte_carlo']['n_simulations'],
    method=config['monte_carlo']['method'],
    save_path=os.path.join(config['output']['plots_dir'], f'{example_ticker}_forecast.png')
)

## 9. Summary and Next Steps

In [ ]:
print("\n" + "="*60)
print("TRAINING PIPELINE COMPLETE")
print("="*60)

print("\n✓ Data loaded and processed")
print("✓ Features engineered")
print("✓ Models trained and evaluated")
print("✓ Models saved")
print("✓ Backtesting framework ready")
print("✓ Monte Carlo simulation demonstrated")

print("\nNext Steps:")
print("1. Run full walk-forward backtest")
print("2. Optimize hyperparameters if needed")
print("3. Train models for other horizons (63d, 126d)")
print("4. Use CLI for predictions: python cli.py predict --ticker BBCA.JK")
print("\nFor CLI usage: python cli.py --help")